# Complexity-conditioned GFlowNet：真实 6/15 Smoke

> **历史诊断归档：禁止恢复或重新运行。** 本 Notebook 依赖已移除的 anchor/checkpoint v5 架构，仅用于审计旧结果；新的正式训练必须使用 no-anchor Notebook 与 `factor_gfn.checkpoint.no_anchor.v1`。

这是第 10 步的短兼容性检查，不是正式训练。它只读取冻结训练期数据，自动完成 N=1/2 exhaustive、exact Z、training-only calibration、两次 discovery+anchor 更新和 checkpoint deterministic resume。

默认安全锁 `RUN_SMOKE=False`。先运行环境与配置单元，确认路径、CUDA 和预算，再人工改为 `True`。若出现结构性错误、无有效 exhaustive candidate、calibration 样本不足、非有限 TB、严重 N 层有效率失衡或 checkpoint 不一致，应立即停止，不修改 Reward 或放宽行业中性化。

In [ ]:
import json
import math
import platform
import sys
import warnings
from dataclasses import asdict
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_dir, *working_dir.parents) if (path / 'factor_gfn').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('无法从当前目录向上找到 factor_gfn 项目根目录')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from factor_gfn.barra import STYLE_NAMES
from factor_gfn.gfn import (
    DEFAULT_REAL_REWARD_CONFIG,
    ComplexitySchedulerConfig,
    ExhaustiveAnchorConfig,
    ExhaustiveAnchorPool,
    ExhaustivePlanningConfig,
    ExhaustiveRegistry,
    GFNConfig,
    GFNTrainer,
    ModelConfig,
    NormalizerCalibrationConfig,
    RealRewardDataPaths,
    RealRewardProvider,
    SamplingConfig,
    StateAdapter,
    TrainingConfig,
    build_real_reward_data_context,
    resolve_exhaustive_plan,
)
from factor_gfn.grammar import Expression, SearchSpaceConfig


## 1. 人工配置与安全锁

`SMOKE_RUN_NAME` 是可恢复目录名。中断后使用同一个名称重跑，SQLite exhaustive registry 会从 pending candidates 继续，不删除已有结果。不要复用正式 run 目录。

In [ ]:
RUN_SMOKE = False  # 完成下方只读预检后，人工改为 True
DEVICE = 'cuda:0'
SEED = 42
SMOKE_RUN_NAME = 'manual_smoke_6_15_seed42'
MAX_DEPTH = 6
MAX_NODES = 15
BATCH_SIZE = 8
DISCOVERY_STEPS = 2  # 第1步后保存；第2步用于确定性恢复对照
CALIBRATION_MIN_VALID_PER_N = 1
CALIBRATION_MAX_REQUESTED_PER_N = 2
EXACT_NODE_RETRY_BUDGET = 2
ANCHOR_FREQUENCY = 1
ANCHOR_BATCH_SIZE = 8
EXHAUSTIVE_COUNTS = (1, 2)
SMOKE_ROOT = PROJECT_ROOT / 'runs' / 'complexity_smoke_6_15' / SMOKE_RUN_NAME
REGISTRY_PATH = SMOKE_ROOT / 'exhaustive_registry.sqlite3'
CHECKPOINT_PATH = SMOKE_ROOT / 'checkpoint_after_discovery_1.pt'

if DEVICE != 'cuda:0' and not DEVICE.startswith('cuda:'):
    raise ValueError('真实 smoke 必须显式使用 CUDA device')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA 不可用；不要回落 CPU 执行真实 smoke')
device = torch.device(DEVICE)
torch.cuda.set_device(device)

data_paths = RealRewardDataPaths()
barra_paths = data_paths.barra_paths
required_paths = [
    data_paths.tensor_path, data_paths.universe_mask_path,
    data_paths.date_list_path, data_paths.stock_list_path,
    data_paths.processed_metadata_path, data_paths.industry_path,
    data_paths.industry_metadata_path, barra_paths.metadata_path,
    barra_paths.market_return_path,
    *[barra_paths.exposure_path(name) for name in STYLE_NAMES],
]
input_frame = pd.DataFrame([{
    'path': str(path), 'exists': path.is_file(),
    'size_mib': path.stat().st_size / 1024**2 if path.is_file() else np.nan,
} for path in required_paths])
display(input_frame)
missing = input_frame.loc[~input_frame['exists'], 'path'].tolist()
if missing:
    raise FileNotFoundError(f'真实 Reward 输入缺失：{missing}')
display(pd.DataFrame([{
    'project_root': str(PROJECT_ROOT), 'smoke_root': str(SMOKE_ROOT),
    'python': platform.python_version(), 'torch': torch.__version__,
    'device': str(device), 'gpu': torch.cuda.get_device_name(device),
    'run_smoke': RUN_SMOKE,
}]))
print('只读预检完成。确认上述路径和 GPU 后，再将 RUN_SMOKE 改为 True。')


## 2. 构建真实 training-only context、Provider 与 6/15 配置

本单元开始加载真实数据。不会加载 validation/OOS；不会改变 Reward、行业中性化或固定调仓日历。

In [ ]:
if not RUN_SMOKE:
    raise RuntimeError('安全锁仍为 False；确认预算后再人工开启')
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
phase_times = {}
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)
total_started = perf_counter()

started = perf_counter()
context = build_real_reward_data_context()
phase_times['build_real_reward_context'] = perf_counter() - started
assert context.manifest['calendar']['last_rebalance_date'] <= '2018-12-31'

started = perf_counter()
provider = RealRewardProvider(context, DEFAULT_REAL_REWARD_CONFIG)
phase_times['build_real_reward_provider'] = perf_counter() - started
provider_manifest = provider.manifest()
assert provider_manifest['data_scope'] == 'training_only'
assert provider_manifest['validation_oos_loaded'] is False
assert provider_manifest['industry_neutralization']['enabled'] is True
assert provider.reward_config.candidate_industry_neutralization is True

search_space = SearchSpaceConfig(max_depth=MAX_DEPTH, max_nodes=MAX_NODES)
config = GFNConfig(
    search_space=search_space,
    model=ModelConfig(
        d_model=128, num_heads=4, num_layers=4,
        dim_feedforward=512, dropout=0.0,
        token_policy_mode='grammar_hierarchical',
    ),
    sampling=SamplingConfig(temperature=1.0, greedy=False),
    reward=DEFAULT_REAL_REWARD_CONFIG,
    training=TrainingConfig(
        batch_size=BATCH_SIZE, learning_rate=1e-4,
        log_z_learning_rate=1e-2, initial_log_z=39.0,
        max_steps=DISCOVERY_STEPS, model_gradient_clip_norm=5.0,
        log_z_gradient_clip_norm=5.0, deterministic_algorithms=True, seed=SEED,
    ),
    complexity_scheduler=ComplexitySchedulerConfig(
        enabled=True, exhaustive_node_counts=EXHAUSTIVE_COUNTS,
        exact_node_retry_budget=EXACT_NODE_RETRY_BUDGET,
        low_effective_update_rate_warning_threshold=0.25,
    ),
    calibration=NormalizerCalibrationConfig(
        enabled=True,
        minimum_valid_calibration_samples=CALIBRATION_MIN_VALID_PER_N,
        maximum_requested_calibration_slots_per_N=CALIBRATION_MAX_REQUESTED_PER_N,
    ),
    exhaustive_anchors=ExhaustiveAnchorConfig(
        enabled=True, frequency=ANCHOR_FREQUENCY,
        batch_size=ANCHOR_BATCH_SIZE, seed_offset=1,
    ),
)
resolved = config.resolved_complexity_strata()
assert resolved['resolved_exhaustive_node_counts'] == (1, 2)
assert set(resolved['resolved_discovery_node_counts']).isdisjoint({1, 2})
display(pd.DataFrame([{
    'F': resolved['resolved_feasible_node_counts'],
    'E': resolved['resolved_exhaustive_node_counts'],
    'S': resolved['resolved_discovery_node_counts'],
    'config_fingerprint': config.fingerprint(),
    'provider_fingerprint': provider.fingerprint(),
    'context_fingerprint': context.fingerprint,
}]))


## 3. 可恢复的 N=1/2 exhaustive RealReward 与 exact Z

SQLite 每条评价立即提交。若中断，保留目录并重新运行，本单元只处理 pending candidates。N≥3 被显式排除在 smoke exhaustive 之外。

In [ ]:
started = perf_counter()
planning = ExhaustivePlanningConfig(
    canonical_count_cap=10_000,
    estimated_real_reward_seconds_per_candidate=0.75,
    planned_real_reward_budget_seconds=3600.0,
    max_budget_fraction=0.20,
    explicit_include_node_counts=(1, 2),
    explicit_exclude_node_counts=tuple(range(3, MAX_NODES + 1)),
)
plan = resolve_exhaustive_plan(search_space, planning)
assert plan.resolved_exhaustive_node_counts == (1, 2)
registry = ExhaustiveRegistry(REGISTRY_PATH)
registry.register_plan(
    plan, provider_fingerprint=provider.fingerprint(),
    context_fingerprint=context.fingerprint,
)
phase_times['exhaustive_plan_and_registration'] = perf_counter() - started

exhaustive_progress = []
started = perf_counter()
for node_count in EXHAUSTIVE_COUNTS:
    pending = registry.pending_candidates(node_count)
    print(f'N={node_count}: pending={len(pending)}, coverage={registry.coverage(node_count)}')
    for index, candidate in enumerate(pending, start=1):
        assignment = provider.evaluate(
            Expression.from_prefix(candidate.prefix_token_ids)
        )
        registry.record_evaluation(
            candidate.structural_hash, valid=assignment.valid,
            reward_details=assignment.metadata or {},
            rejection_reason=assignment.rejection_reason,
            target_mass=assignment.reward if assignment.valid else 0.0,
        )
        if index % 25 == 0 or index == len(pending):
            row = {'N': node_count, 'finished_this_run': index, **registry.coverage(node_count)}
            exhaustive_progress.append(row)
            print(row)
phase_times['exhaustive_real_reward'] = perf_counter() - started

exact_results = {}
exact_rows = []
for node_count in EXHAUSTIVE_COUNTS:
    result = registry.compute_exact_masses(
        node_count, reward_floor=config.reward.reward_floor
    )
    exact_results[node_count] = result
    exact_rows.append({**asdict(result), **registry.coverage(node_count)})
exact_frame = pd.DataFrame(exact_rows).set_index('node_count')
display(exact_frame)
assert exact_frame['coverage_complete'].all()
assert np.isfinite(exact_frame['exact_tb_log_z'].astype(float)).all()


## 4. Trainer、exact buffer、anchor pool 与 training-only calibration

Smoke 默认每个 feasible N 只要求 1 个有效 calibration 样本，最多请求 2 个。这只是链路检查，不是正式的 64/128 校准标准。

In [ ]:
started = perf_counter()
trainer = GFNTrainer(config, provider, device=device)
for result in exact_results.values():
    trainer.register_exact_mass_result(result)
anchor_pool = ExhaustiveAnchorPool.from_registry(
    registry, node_counts=EXHAUSTIVE_COUNTS, adapter=trainer.adapter
)
trainer.configure_exhaustive_anchor_pool(anchor_pool)
phase_times['trainer_and_anchor_pool'] = perf_counter() - started

started = perf_counter()
calibration_result = None
while calibration_result is None:
    calibration_result = trainer.calibration_step()
phase_times['training_only_calibration'] = perf_counter() - started
calibration_frame = pd.DataFrame(
    [asdict(value) for value in calibration_result.values()]
).set_index('node_count').sort_index()
display(calibration_frame)
assert trainer.optimizer_step == 0
assert trainer.total_policy_optimizer_step == 0
assert trainer.calibration.status == 'complete'
assert trainer.complexity_scheduler.state_dict()['position'] == 0
assert provider_manifest['validation_oos_loaded'] is False


## 5. 两次 discovery+anchor 与 checkpoint deterministic resume

TB hook 只观察每次 forward 的轨迹、delta、Reward 与 depth。N=1/2 调用标记为 anchor，N∈S 标记为 discovery。它不修改梯度、采样或优化器。

In [ ]:
def install_tb_capture(target_trainer, target_rows, branch):
    exhaustive_set = set(target_trainer.resolved_exhaustive_node_counts)
    def hook(module, inputs, output):
        trajectories = inputs[0]
        for trajectory, delta in zip(trajectories, output.deltas.detach().cpu().tolist(), strict=True):
            node_count = int(trajectory.target_node_count)
            target_rows.append({
                'branch': branch,
                'source': 'anchor' if node_count in exhaustive_set else 'discovery',
                'N': node_count,
                'delta': float(delta),
                'reward': float(trajectory.reward),
                'log_reward': float(trajectory.log_reward),
                'depth': int(trajectory.terminal_expression.stats.depth),
                'node_count': int(trajectory.terminal_expression.stats.node_count),
                'structural_hash': trajectory.terminal_expression.structural_hash(),
            })
    return target_trainer.tb_loss.register_forward_hook(hook)

tb_rows = []
capture = install_tb_capture(trainer, tb_rows, 'source')
started = perf_counter()
first_stats = trainer.train_step()
phase_times['discovery_anchor_step_1'] = perf_counter() - started
assert not first_stats.skipped_update
assert trainer.optimizer_step == 1
assert trainer.anchor_optimizer_step == 1
assert trainer.total_policy_optimizer_step == 2
trainer.save_checkpoint(CHECKPOINT_PATH)

started = perf_counter()
expected_stats = trainer.train_step()
phase_times['discovery_anchor_step_2_continuous'] = perf_counter() - started
expected_model = {key: value.detach().cpu().clone() for key, value in trainer.model.state_dict().items()}
expected_discovery_state = trainer.complexity_scheduler.state_dict()
expected_anchor_state = trainer.anchor_sampler.state_dict()
capture.remove()

resumed_provider = provider  # 同一冻结真实 context；Reward 数值合同不变
resumed = GFNTrainer(config, resumed_provider, device=device)
for result in exact_results.values():
    resumed.register_exact_mass_result(result)
resumed.configure_exhaustive_anchor_pool(anchor_pool)
resume_rows = []
resume_capture = install_tb_capture(resumed, resume_rows, 'resumed')
resumed.load_checkpoint(CHECKPOINT_PATH)
started = perf_counter()
actual_stats = resumed.train_step()
phase_times['discovery_anchor_step_2_resumed'] = perf_counter() - started
resume_capture.remove()

assert asdict(actual_stats) == asdict(expected_stats)
for key, expected in expected_model.items():
    torch.testing.assert_close(resumed.model.state_dict()[key].detach().cpu(), expected, rtol=0, atol=0)
assert resumed.complexity_scheduler.state_dict() == expected_discovery_state
assert resumed.anchor_sampler.state_dict() == expected_anchor_state
assert resumed.optimizer_step == 2
assert resumed.anchor_optimizer_step == 2
assert resumed.total_policy_optimizer_step == 4
print('CHECKPOINT_RESUME_EXACT_OK')


## 6. 汇总逐 N 诊断并落盘

重点人工查看：某个 discovery N 是否持续无效、retry exhausted、delta 非有限或极端，anchor loss 是否有限，行业中性化是否仍启用，以及恢复是否严格一致。这里只做短 smoke，不据此冻结 calibration 样本量或训练超参数。

In [ ]:
phase_times['total_notebook_smoke'] = perf_counter() - total_started
tb_frame = pd.DataFrame(tb_rows)
resume_tb_frame = pd.DataFrame(resume_rows)
display(tb_frame)

per_n_tb = (
    tb_frame.groupby(['source', 'N'], as_index=False)
    .agg(
        trajectories=('N', 'size'),
        delta_mean=('delta', 'mean'), delta_std=('delta', 'std'),
        delta_abs_max=('delta', lambda values: float(np.abs(values).max())),
        reward_mean=('reward', 'mean'), reward_min=('reward', 'min'),
        reward_max=('reward', 'max'), depth_mean=('depth', 'mean'),
        depth_min=('depth', 'min'), depth_max=('depth', 'max'),
    )
)
counter_rows = []
rates = resumed._effective_update_rate_by_N()
for node_count in resumed.resolved_discovery_node_counts:
    counter_rows.append({
        'N': node_count,
        'requested': resumed.requested_count_by_N[node_count],
        'sampled_attempts': resumed.sampled_attempt_count_by_N[node_count],
        'valid': resumed.valid_count_by_N[node_count],
        'successful': resumed.successful_update_count_by_N[node_count],
        'effective_update_rate': rates[node_count],
        'retry_exhausted': resumed.retry_exhausted_count_by_N[node_count],
    })
counter_frame = pd.DataFrame(counter_rows).set_index('N')
anchor_counter_frame = pd.DataFrame({
    'requested': resumed.anchor_sampler.requested_count_by_N,
    'sampled': resumed.anchor_sampler.sampled_count_by_N,
    'successful': resumed.anchor_sampler.successful_update_count_by_N,
}).rename_axis('N')
phase_frame = pd.DataFrame([{'phase': key, 'wall_seconds': value} for key, value in phase_times.items()])
gpu_frame = pd.DataFrame([{
    'device': str(device), 'gpu': torch.cuda.get_device_name(device),
    'allocated_mib': torch.cuda.memory_allocated(device) / 1024**2,
    'reserved_mib': torch.cuda.memory_reserved(device) / 1024**2,
    'peak_allocated_mib': torch.cuda.max_memory_allocated(device) / 1024**2,
    'peak_reserved_mib': torch.cuda.max_memory_reserved(device) / 1024**2,
}])

display(per_n_tb)
display(counter_frame)
display(anchor_counter_frame)
display(phase_frame)
display(gpu_frame)
display(pd.DataFrame([asdict(first_stats), asdict(expected_stats)]))

acceptance = {
    'resolved_E_is_1_2': resolved['resolved_exhaustive_node_counts'] == (1, 2),
    'exact_coverage_complete': bool(exact_frame['coverage_complete'].all()),
    'exact_log_z_finite': bool(np.isfinite(exact_frame['exact_tb_log_z']).all()),
    'calibration_complete': trainer.calibration.status == 'complete',
    'discovery_updates_two': resumed.optimizer_step == 2,
    'anchor_updates_two': resumed.anchor_optimizer_step == 2,
    'total_policy_updates_four': resumed.total_policy_optimizer_step == 4,
    'tb_deltas_finite': bool(np.isfinite(tb_frame['delta']).all()),
    'anchor_loss_finite': bool(np.isfinite([first_stats.anchor_loss, expected_stats.anchor_loss]).all()),
    'industry_neutralization_on': bool(
        provider.manifest()['industry_neutralization']['enabled']
        and provider.reward_config.candidate_industry_neutralization
    ),
    'training_only': provider_manifest['data_scope'] == 'training_only',
    'validation_oos_not_loaded': provider_manifest['validation_oos_loaded'] is False,
    'checkpoint_resume_exact': asdict(actual_stats) == asdict(expected_stats),
}
acceptance_frame = pd.DataFrame([acceptance])
display(acceptance_frame.T.rename(columns={0: 'passed'}))

exact_frame.to_csv(SMOKE_ROOT / 'exact_strata.csv')
calibration_frame.to_csv(SMOKE_ROOT / 'calibration_by_n.csv')
tb_frame.to_csv(SMOKE_ROOT / 'trajectory_diagnostics.csv', index=False)
per_n_tb.to_csv(SMOKE_ROOT / 'tb_reward_depth_by_n.csv', index=False)
counter_frame.to_csv(SMOKE_ROOT / 'discovery_counters_by_n.csv')
anchor_counter_frame.to_csv(SMOKE_ROOT / 'anchor_counters_by_n.csv')
phase_frame.to_csv(SMOKE_ROOT / 'wall_time_by_phase.csv', index=False)
gpu_frame.to_csv(SMOKE_ROOT / 'gpu_memory.csv', index=False)
summary = {
    'schema': 'factor_gfn.complexity_smoke_6_15.v1',
    'config_manifest': config.manifest(),
    'provider_fingerprint': provider.fingerprint(),
    'context_fingerprint': context.fingerprint,
    'resolved_complexity': resolved,
    'phase_times': phase_times,
    'gpu': gpu_frame.iloc[0].to_dict(),
    'acceptance': acceptance,
    'first_stats': asdict(first_stats),
    'second_stats': asdict(expected_stats),
}
(SMOKE_ROOT / 'smoke_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
registry.close()
assert all(acceptance.values()), acceptance
print('SMOKE_ACCEPTANCE_OK')
print('结果目录：', SMOKE_ROOT)
print('停止：不要在 6/15 上增加训练步数。把本目录结果交给 Codex 分析。')


In [ ]:
current_provider_manifest = provider.manifest()

assert current_provider_manifest[
    'industry_neutralization'
]['enabled'] is True
assert provider.reward_config.candidate_industry_neutralization is True

acceptance['industry_neutralization_on'] = True
assert all(acceptance.values()), acceptance

summary_path = SMOKE_ROOT / 'smoke_summary.json'
saved_summary = json.loads(summary_path.read_text(encoding='utf-8'))
saved_summary['acceptance'] = acceptance
summary_path.write_text(
    json.dumps(saved_summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('SMOKE_ACCEPTANCE_OK')
print('结果目录：', SMOKE_ROOT)